# Estudo Comparativo de Classificação Acústica Submarina — Dataset IARA
### Notebook 11: Comparativo Multimodelo com o Extrator HHT (Random Forest, MLP, SVM e CNN)

Neste notebook, investigamos sistematicamente o comportamento da **Transformada de Hilbert-Huang (HHT Pura)** ($256\text{ bins}$ de frequência instantânea, de $1{,}0\text{ Hz}$ a $1000\text{ Hz}$) através de **quatro famílias distintas de aprendizado de máquina**:

1. **Random Forest (100 árvores de decisão):** Ensembles baseados em particionamento ortogonal de espaço com balanceamento de pesos de classe.
2. **Multi-Layer Perceptron (MLP PyTorch `[32, 16]`):** Rede neural feedforward com `BatchNorm1d` e `Dropout 0.2` treinada na GPU via PyTorch.
3. **SVM Nyström Gaussiano ($m=6000$):** Núcleo RBF aproximado em espaço dimensional de landmark points com penalidade ElasticNet.
4. **Convolutional Neural Network (CNN 2D PyTorch `[64, 32]`):** Convoluções espaciais $2\text{D}$ em janelas temporais de espectrograma Hilbert-Huang ($32 \times 256$) na GPU **NVIDIA GeForce RTX 5060 Ti** via container Docker.

Todos os modelos foram avaliados sob o protocolo oficial de **$5\times2$ validação cruzada (10 folds)** com particionamento estrito por `Ship ID` sobre todos os **1.706 áudios de UO** do dataset IARA (*Silva et al.*, IEEE Access 2025), utilizando votação majoritária por arquivo (*Majority Voting*).


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display, HTML
except ImportError:
    display = print

# Configuração de estilo visual acadêmico premium
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 15,
    'figure.dpi': 120
})

print('Ambiente configurado com sucesso! Bibliotecas prontas.')


## 1. Tabela Mestra Consolidada: HHT vs. DEMON vs. Artigo Original

Abaixo confrontamos diretamente o desempenho da **HHT Pura (256-D)** em cada modelo contra as representações espectrais diretas do artigo original (**MEL** e **LOFAR**) e o extrator de envoltória **DEMON**:


In [ ]:
tabela_dados = [
    # Baselines do Artigo Original (Silva et al., 2025)
    {'Modelo': 'MLP (Silva et al., 2025)', 'Rep': 'LOFAR', 'Dim': '1024', 'Acc 4C (%)': 67.48, 'SP 4C (%)': 66.51, 'Rec SMALL (%)': np.nan, 'Rec MED (%)': np.nan, 'Rec LARGE (%)': np.nan, 'Rec AMB (%)': np.nan, 'Acc 3C (%)': np.nan, 'SP 3C (%)': np.nan},
    {'Modelo': 'CNN (Silva et al., 2025)', 'Rep': 'LOFAR', 'Dim': '32x1024', 'Acc 4C (%)': 67.02, 'SP 4C (%)': 66.05, 'Rec SMALL (%)': np.nan, 'Rec MED (%)': np.nan, 'Rec LARGE (%)': np.nan, 'Rec AMB (%)': np.nan, 'Acc 3C (%)': np.nan, 'SP 3C (%)': np.nan},
    {'Modelo': 'SVM m=6000 (Proposto)', 'Rep': 'LOFAR', 'Dim': '1024', 'Acc 4C (%)': 65.47, 'SP 4C (%)': 63.83, 'Rec SMALL (%)': 47.81, 'Rec MED (%)': 55.77, 'Rec LARGE (%)': 74.91, 'Rec AMB (%)': 77.65, 'Acc 3C (%)': 59.41, 'SP 3C (%)': 60.59},
    {'Modelo': 'SVM Golden (Proposto)', 'Rep': 'MEL', 'Dim': '256', 'Acc 4C (%)': 64.56, 'SP 4C (%)': 63.78, 'Rec SMALL (%)': 50.90, 'Rec MED (%)': 60.40, 'Rec LARGE (%)': 69.16, 'Rec AMB (%)': 77.86, 'Acc 3C (%)': 61.20, 'SP 3C (%)': 60.50},
    {'Modelo': 'CNN (Silva et al., 2025)', 'Rep': 'MEL', 'Dim': '32x256', 'Acc 4C (%)': 64.99, 'SP 4C (%)': 63.52, 'Rec SMALL (%)': np.nan, 'Rec MED (%)': np.nan, 'Rec LARGE (%)': np.nan, 'Rec AMB (%)': np.nan, 'Acc 3C (%)': np.nan, 'SP 3C (%)': np.nan},
    {'Modelo': 'MLP (Silva et al., 2025)', 'Rep': 'MEL', 'Dim': '256', 'Acc 4C (%)': 64.51, 'SP 4C (%)': 63.38, 'Rec SMALL (%)': np.nan, 'Rec MED (%)': np.nan, 'Rec LARGE (%)': np.nan, 'Rec AMB (%)': np.nan, 'Acc 3C (%)': np.nan, 'SP 3C (%)': np.nan},
    {'Modelo': 'RF (Silva et al., 2025)', 'Rep': 'MEL', 'Dim': '256', 'Acc 4C (%)': 62.64, 'SP 4C (%)': 62.22, 'Rec SMALL (%)': np.nan, 'Rec MED (%)': np.nan, 'Rec LARGE (%)': np.nan, 'Rec AMB (%)': np.nan, 'Acc 3C (%)': np.nan, 'SP 3C (%)': np.nan},
    {'Modelo': 'RF (Silva et al., 2025)', 'Rep': 'LOFAR', 'Dim': '1024', 'Acc 4C (%)': 58.87, 'SP 4C (%)': 56.92, 'Rec SMALL (%)': np.nan, 'Rec MED (%)': np.nan, 'Rec LARGE (%)': np.nan, 'Rec AMB (%)': np.nan, 'Acc 3C (%)': np.nan, 'SP 3C (%)': np.nan},
    # Modelos Avaliados com HHT Pura (10 Folds 5x2cv)
    {'Modelo': 'Random Forest 100t (Campeão HHT)', 'Rep': 'HHT', 'Dim': '256', 'Acc 4C (%)': 59.94, 'SP 4C (%)': 58.14, 'Rec SMALL (%)': 54.83, 'Rec MED (%)': 46.64, 'Rec LARGE (%)': 73.83, 'Rec AMB (%)': 59.08, 'Acc 3C (%)': 60.43, 'SP 3C (%)': 58.27},
    {'Modelo': 'MLP PyTorch [32, 16] (GPU)', 'Rep': 'HHT', 'Dim': '256', 'Acc 4C (%)': 56.37, 'SP 4C (%)': 56.14, 'Rec SMALL (%)': 28.15, 'Rec MED (%)': 56.85, 'Rec LARGE (%)': 68.41, 'Rec AMB (%)': 80.61, 'Acc 3C (%)': 59.70, 'SP 3C (%)': 58.76},
    {'Modelo': 'SVM Nyström m=6000', 'Rep': 'HHT', 'Dim': '256', 'Acc 4C (%)': 55.63, 'SP 4C (%)': 53.63, 'Rec SMALL (%)': 44.49, 'Rec MED (%)': 44.30, 'Rec LARGE (%)': 67.15, 'Rec AMB (%)': 67.96, 'Acc 3C (%)': 56.93, 'SP 3C (%)': 52.82},
    {'Modelo': 'CNN 2D PyTorch (GPU)', 'Rep': 'HHT', 'Dim': '32x256', 'Acc 4C (%)': 46.92, 'SP 4C (%)': 46.10, 'Rec SMALL (%)': 21.80, 'Rec MED (%)': 48.32, 'Rec LARGE (%)': 54.81, 'Rec AMB (%)': 73.16, 'Acc 3C (%)': 52.35, 'SP 3C (%)': 49.82},
    # Melhor Modelo com DEMON (Notebook 7)
    {'Modelo': 'MLP PyTorch [32, 16]', 'Rep': 'DEMON', 'Dim': '253', 'Acc 4C (%)': 52.10, 'SP 4C (%)': 52.12, 'Rec SMALL (%)': 29.78, 'Rec MED (%)': 46.04, 'Rec LARGE (%)': 62.66, 'Rec AMB (%)': 78.78, 'Acc 3C (%)': 57.12, 'SP 3C (%)': 56.00}
]

df_master = pd.DataFrame(tabela_dados)
display(df_master)


## 2. Comparativo Global: O Desempenho da HHT Pura (4 Classes vs. 3 Classes)

No gráfico abaixo, comparamos a **Acurácia Global** e o **Índice SP Oficial** de cada uma das 4 arquiteturas no problema geral de 4 Classes e no problema focal de 3 Classes (apenas embarcações), confrontando com os tetos de referência de LOFAR e MEL:


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

hht_models = ['Random Forest (100t)', 'MLP [32, 16]', 'SVM (m=6000)', 'CNN 2D (GPU)']
sp_4c = [58.14, 56.14, 53.63, 46.10]
sp_4c_err = [1.79, 1.52, 4.40, 4.14]
acc_4c = [59.94, 56.37, 55.63, 46.92]
acc_4c_err = [1.48, 1.29, 2.81, 2.13]

sp_3c = [58.27, 58.76, 52.82, 49.82]
sp_3c_err = [1.86, 2.33, 5.48, 2.73]
acc_3c = [60.43, 59.70, 56.93, 52.35]
acc_3c_err = [1.53, 2.08, 3.24, 2.32]

x = np.arange(len(hht_models))
w = 0.35

# Gráfico 1: 4 Classes
rects1 = ax1.bar(x - w/2, acc_4c, w, yerr=acc_4c_err, capsize=4, label='Acurácia Global (%)', color='#2980b9', alpha=0.9)
rects2 = ax1.bar(x + w/2, sp_4c, w, yerr=sp_4c_err, capsize=4, label='Índice SP Oficial (%)', color='#9b59b6', alpha=0.9)
ax1.axhline(66.51, color='#27ae60', linestyle='--', lw=2, label='Referência LOFAR MLP (66.51%)')
ax1.axhline(63.78, color='#e67e22', linestyle=':', lw=2, label='Referência MEL SVM (63.78%)')
ax1.axhline(52.12, color='#95a5a6', linestyle='-.', lw=1.5, label='Melhor DEMON MLP (52.12%)')
ax1.set_title('Problema Geral de 4 Classes (com AMBIENTAL)', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(hht_models, rotation=20, ha='right', fontweight='bold')
ax1.set_ylabel('Taxa (%)')
ax1.set_ylim(35, 75)
ax1.legend(loc='lower left', frameon=True, fontsize=9)

for rect in rects2:
    h = rect.get_height()
    ax1.annotate(f'{h:.1f}%', xy=(rect.get_x() + rect.get_width()/2, h),
                 xytext=(0, 7), textcoords="offset points", ha='center', va='bottom', fontsize=9.5, fontweight='bold', color='#6c3483')

# Gráfico 2: 3 Classes
rects3 = ax2.bar(x - w/2, acc_3c, w, yerr=acc_3c_err, capsize=4, label='Acurácia (Embarcações)', color='#27ae60', alpha=0.9)
rects4 = ax2.bar(x + w/2, sp_3c, w, yerr=sp_3c_err, capsize=4, label='Índice SP (Embarcações)', color='#d35400', alpha=0.9)
ax2.axhline(60.59, color='#2c3e50', linestyle='--', lw=2, label='Referência LOFAR SVM (60.59%)')
ax2.axhline(56.00, color='#95a5a6', linestyle='-.', lw=1.5, label='Melhor DEMON MLP (56.00%)')
ax2.set_title('Problema Focal de 3 Classes (Apenas Embarcações)', fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(hht_models, rotation=20, ha='right', fontweight='bold')
ax2.set_ylabel('Taxa (%)')
ax2.set_ylim(35, 75)
ax2.legend(loc='lower left', frameon=True, fontsize=9)

for rect in rects4:
    h = rect.get_height()
    ax2.annotate(f'{h:.1f}%', xy=(rect.get_x() + rect.get_width()/2, h),
                 xytext=(0, 7), textcoords="offset points", ha='center', va='bottom', fontsize=9.5, fontweight='bold', color='#ba4a00')

plt.suptitle('Comparativo Direto de Modelos com a HHT Pura (10 Folds Oficiais 5x2cv)', fontsize=15, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


### Principais Observações do Comparativo Global:
1. **Random Forest Lidera em 4 Classes ($58{,}14\%$ SP):** Supera o Random Forest no LOFAR ($56{,}92\%$) e supera amplamente qualquer modelo com DEMON isolado ($52{,}12\%$).
2. **MLP Lidera em 3 Classes ($58{,}76\%$ SP):** Ao excluir o ruído de fundo, a representação de Hilbert-Huang permite que o MLP se aproxime do desempenho de topo do LOFAR ($60{,}59\%$).
3. **CNN 2D Sofre Queda Acentuada ($46{,}10\%$ SP):** A convolução $2\text{D}$ com pooling borra as raias de frequências instantâneas altamente localizadas das IMFs, confirmando que a HHT se beneficia mais de modelos densos ou tabulares do que de visão computacional clássica.


## 3. Análise de Sensibilidade Foco-por-Classe (Per-Class Recalls)

Abaixo avaliamos o perfil de sensibilidade de cada modelo em cada uma das classes do problema geral de 4 classes com HHT:


In [ ]:
classes = ['SMALL', 'MEDIUM', 'LARGE', 'AMBIENTAL']
models_subset = ['Random Forest', 'MLP [32, 16]', 'SVM (m=6000)', 'CNN 2D (GPU)']

recalls_matrix = np.array([
    [54.83, 46.64, 73.83, 59.08],  # Random Forest
    [28.15, 56.85, 68.41, 80.61],  # MLP
    [44.49, 44.30, 67.15, 67.96],  # SVM m=6000
    [21.80, 48.32, 54.81, 73.16]   # CNN 2D
])

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(classes))
width = 0.20
colors = ['#27ae60', '#3498db', '#8e44ad', '#e74c3c']

for i, (m_name, color) in enumerate(zip(models_subset, colors)):
    offset = (i - 1.5) * width
    rects = ax.bar(x + offset, recalls_matrix[i], width, label=m_name, color=color, alpha=0.9)
    for rect in rects:
        h = rect.get_height()
        ax.annotate(f'{h:.1f}%', xy=(rect.get_x() + rect.get_width()/2, h),
                    xytext=(0, 2), textcoords="offset points", ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax.set_title('Sensibilidade por Classe (Recall %) no Problema de 4 Classes com HHT Pura', fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(classes, fontsize=12, fontweight='bold')
ax.set_ylabel('Recall (%)', fontsize=12)
ax.set_ylim(0, 95)
ax.legend(title='Arquitetura', loc='upper left', frameon=True)
ax.grid(True, axis='y', alpha=0.4)

plt.tight_layout()
plt.show()


### Diagnóstico Físico da Sensibilidade por Classe:
1. **Random Forest e a Liderança em SMALL ($54{,}83\%$):** O particionamento em árvores consegue capturar transientes de alta frequência instantânea típicos de lanchas sem diluição por médias globais.
2. **MLP é Excelente em AMBIENTAL ($80{,}61\%$):** A camada `BatchNorm1d` normaliza a densidade espectral marginal contínua e separa eficazmente o ruído gaussiano de fundo.
3. **Detecção Consistente de LARGE ($67\%$ a $74\%$):** O pico de sub-harmônicos lentos em $\sim 18\text{ Hz}$ e a alta energia de maquinário em $450\text{--}550\text{ Hz}$ garantem que navios de grande porte sejam corretamente identificados por quase todas as arquiteturas.


## 4. Matrizes de Confusão Acumuladas nos 10 Folds ($5\times2$cv)

### 4.1 Problema Geral de 4 Classes (SMALL, MEDIUM, LARGE, AMBIENTAL)
Abaixo plotamos lado a lado os mapas de calor de confusão normalizados das 4 arquiteturas avaliadas (totalizando 17.060 predições de áudio):


In [ ]:
# Matrizes de Confusão 4-Classes Normalizadas (%)
cm_rf_4c = np.array([
    [54.8, 10.4, 32.3,  2.5],
    [34.1, 46.6, 18.5,  0.8],
    [19.3,  6.1, 73.8,  0.8],
    [26.8,  5.3,  8.8, 59.1]
])

cm_mlp_4c = np.array([
    [28.1, 21.3, 30.2, 20.3],
    [17.9, 56.8, 13.0, 12.3],
    [12.9, 11.3, 68.4,  7.4],
    [ 8.3,  3.9,  7.2, 80.6]
])

cm_svm_4c = np.array([
    [44.5, 10.6, 28.9, 16.0],
    [29.3, 44.3, 14.8, 11.6],
    [19.2,  7.1, 67.1,  6.5],
    [17.8,  5.1,  9.2, 68.0]
])

cm_cnn_4c = np.array([
    [21.8, 28.7, 29.6, 19.9],
    [15.6, 48.3, 21.0, 15.1],
    [11.3, 22.9, 54.8, 11.0],
    [ 5.6,  3.2, 18.1, 73.2]
])

cms = [cm_rf_4c, cm_mlp_4c, cm_svm_4c, cm_cnn_4c]
titles = [
    'Random Forest (100t) — SP: 58.14% (Melhor)',
    'MLP PyTorch [32, 16] — SP: 56.14%',
    'SVM Nyström (m=6000) — SP: 53.63%',
    'CNN 2D PyTorch (GPU) — SP: 46.10%'
]
classes = ['SMALL', 'MEDIUM', 'LARGE', 'AMBIENTAL']

fig, axes = plt.subplots(2, 2, figsize=(15, 13))
axes = axes.flatten()

for idx, (cm, title) in enumerate(zip(cms, titles)):
    ax = axes[idx]
    im = ax.imshow(cm, cmap='Purples', vmin=0, vmax=100)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticklabels(classes, fontsize=10, fontweight='bold')
    ax.set_yticklabels(classes, fontsize=10, fontweight='bold')
    ax.set_xlabel('Classe Predita', fontsize=11)
    ax.set_ylabel('Classe Real', fontsize=11)
    
    for i in range(4):
        for j in range(4):
            val = cm[i, j]
            color = 'white' if val > 45 else 'black'
            weight = 'bold' if i == j else 'normal'
            ax.text(j, i, f'{val:.1f}%', ha='center', va='center', color=color, fontsize=11, fontweight=weight)
    
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Taxa (%)', fontsize=10)

plt.suptitle('Matrizes de Confusão Normalizadas — HHT Pura [4 Classes: Geral]', fontsize=15, fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()


### 4.2 Matrizes de Confusão no Problema de 3 Classes (Apenas Embarcações: SMALL, MEDIUM, LARGE)

Ao excluirmos a classe de ruído `AMBIENTAL`, avaliamos a capacidade intrínseca do espectro de Hilbert-Huang de discriminar as tonelagens navais com base na distribuição de frequências instantâneas de máquina e propulsão:


In [ ]:
# Matrizes de Confusão 3-Classes Normalizadas (%)
cm_rf_3c = np.array([
    [55.6, 10.7, 33.7],
    [34.8, 47.3, 17.9],
    [19.5,  6.9, 73.6]
])

cm_mlp_3c = np.array([
    [46.1, 25.4, 28.5],
    [23.2, 64.0, 12.9],
    [17.7, 14.3, 68.1]
])

cm_svm_3c = np.array([
    [39.8, 28.0, 32.2],
    [25.0, 59.6, 15.4],
    [16.2, 14.5, 69.3]
])

cm_cnn_3c = np.array([
    [47.8, 15.2, 37.0],
    [35.0, 41.3, 23.7],
    [25.6, 10.6, 63.8]
])

cms_3c = [cm_rf_3c, cm_mlp_3c, cm_svm_3c, cm_cnn_3c]
titles_3c = [
    'Random Forest (100t) — SP: 58.27%',
    'MLP PyTorch [32, 16] — SP: 58.76% (Melhor)',
    'SVM Nyström (m=6000) — SP: 52.82%',
    'CNN 2D PyTorch (GPU) — SP: 49.82%'
]
classes_3c = ['SMALL', 'MEDIUM', 'LARGE']

fig, axes = plt.subplots(2, 2, figsize=(15, 13))
axes = axes.flatten()

for idx, (cm, title) in enumerate(zip(cms_3c, titles_3c)):
    ax = axes[idx]
    im = ax.imshow(cm, cmap='BuPu', vmin=0, vmax=100)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(classes_3c, fontsize=11, fontweight='bold')
    ax.set_yticklabels(classes_3c, fontsize=11, fontweight='bold')
    ax.set_xlabel('Classe Predita', fontsize=11)
    ax.set_ylabel('Classe Real', fontsize=11)
    
    for i in range(3):
        for j in range(3):
            val = cm[i, j]
            color = 'white' if val > 45 else 'black'
            weight = 'bold' if i == j else 'normal'
            ax.text(j, i, f'{val:.1f}%', ha='center', va='center', color=color, fontsize=12, fontweight=weight)
    
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Taxa (%)', fontsize=10)

plt.suptitle('Matrizes de Confusão Normalizadas — HHT Pura [3 Classes: Apenas Embarcações]', fontsize=15, fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()


### Análise do Padrão de Erro em 3 Classes:
1. **Direção da Confusão de SMALL:** No HHT, a confusão de `SMALL` é majoritariamente direcionada para `LARGE` ($33{,}7\%$ no RF e $37{,}0\%$ na CNN). Isso contrasta diretamente com o **MEL**, onde `SMALL` se confunde quase exclusivamente com `MEDIUM` ($49{,}3\%$).
2. **Complementaridade Perfeita:** Essa discrepância na direção do erro é a base matemática que permite ao **Oráculo atingir $84\%$ de acerto em SMALL** (como demonstrado no Notebook 10), pois quando o MEL erra para `MEDIUM`, o HHT não comete o mesmo erro!


## 5. Diagnóstico Científico e Conclusões Finais

### 5.1 Por que o Random Forest se destacou na HHT ($58{,}14\%$ SP)?
* **Resiliência a *Mode Mixing*:** O processo de peneiramento (*sifting*) do EMD frequentemente divide a energia de harmônicos entre duas IMFs adjacentes. Árvores de decisão realizam cortes ortogonais independentes nos bins de frequência, encontrando regras lógicas simples (ex: se `bin_35 > t1` ou `bin_38 > t2`) sem sofrer a diluição por produto interno de hiperplanos contínuos.
* **Imunidade a Escalas Heterogêneas:** Como cada modo empírico possui variâncias energéticas distintas, modelos baseados em distância (SVM ou CNN) exigem normalizações complexas que podem distorcer picos esparsos. O Random Forest é invariante a transformações monotônicas.

### 5.2 O Confronto HHT vs. DEMON vs. LOFAR
* **HHT supera DEMON Isolado:** Em todas as 4 arquiteturas, a HHT supera o DEMON por uma margem de **$4$ a $14$ pontos percentuais** no Índice SP. Isso ocorre porque a HHT cobre a banda hidroacústica integral ($0\text{--}1000\text{ Hz}$), captando motores auxiliares e ressonâncias de casco, enquanto o DEMON é restrito à banda estreita de rotação de pás ($0\text{--}25\text{ Hz}$).
* **LOFAR e MEL mantêm a liderança linear:** Graças à ortogonalidade exata da FFT e dos bancos triangulares MEL, o LOFAR (via MLP: $66{,}51\%$) e o MEL (via SVM: $63{,}78\%$) continuam como as melhores representações monolíticas.

### 5.3 O Papel Estratégico da HHT no Projeto IARA
1. **Especialista em SMALL:** Conforme revelado no Notebook 10, a HHT Pura é a **líder absoluta em resgates exclusivos da classe SMALL**, salvando sozinha **207 lanchas (11,63%)** que MEL, LOFAR e DEMON perdem simultaneamente.
2. **Candidato Principal para Gating / Dynamic Routing:** Em um sistema multimodal com Mixture of Experts (MoE), a HHT fornece os vetores de desempate ideais para classificar embarcações leves com motores de alta rotação e cascos rápidos.
